# PWM Algorithm Verification — Open In Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/integritynoble/Physics_World_Model/blob/master/examples/Open_In_Colab.ipynb)

**Physics World Model** — 168 imaging modalities, 541 solvers.

This notebook verifies every algorithm for any modality against the benchmark data from [pwm.platformai.org](https://pwm.platformai.org).

**How to use:**
1. Change `MODALITY` in Cell 3 to any of the 168 modalities
2. Run all cells (Runtime → Run All)
3. See PSNR/SSIM results matching the live leaderboard

**All 168 modalities:** `acoustic_emission`, `acoustic_microscopy`, `active_thermography`, `adaptive_optics`, `afm`, `angiography`, `asl_mri`, `atom_probe`, `bioluminescence_tomo`, `brachytherapy_img`, `brillouin`, `cacti`, `cars`, `cassi`, `cathodoluminescence`, `cbct`, `cest_mri`, `ceus`, `clem`, `coded_exposure`, `confocal_3d`, `confocal_endomicroscopy`, `confocal_livecell`, `coronagraphy`, `cryo_em`, `cryo_et`, `ct`, `ct_fluorescence`, `cup`, `dark_field`, `desi`, `dexa`, `dic`, `diffusion_mri`, `digital_breast_tomo`, `dna_paint`, `doppler_ultrasound`, `dot`, `ebsd`, `eddy_current`, `edx_mapping`, `eels`, `eht_imaging`, `elastography`, `electron_diffraction`, `electron_holography`, `electron_tomography`, `endoscopy`, `entangled_photon`, `event_camera`, `expansion`, `fib_sem`, `flash_lidar`, `flim`, `fluoroscopy`, `fmri`, `fpm`, `ftir_imaging`, `fundus`, `fwi`, `gaussian_splatting`, `ghost_imaging`, `gpr`, `gravitational_wave`, `hdr_imaging`, `holography`, `hyperspectral_remote`, `impedance_tomo`, `industrial_ct`, `insar`, `integral`, `ism`, `ivus`, `lattice_lightsheet`, `lensless`, `libs`, `lidar`, `light_field`, `lightsheet`, `lucky_imaging`, `machine_vision`, `magnetic_particle`, `maldi_msi`, `mammography`, `matrix`, `mfm`, `minflux`, `mr_elastography`, `mr_fingerprinting`, `mra`, `mri`, `mrs`, `multispectral_sat`, `muon_tomo`, `nerf`, `neutron_diffraction`, `neutron_tomo`, `nirs_brain`, `nsom`, `ocean_acoustic_tomo`, `ocean_color`, `oct`, `octa`, `odt`, `palm_storm`, `panorama`, `particle_calorimetry`, `passive_microwave`, `pet`, `pet_ct`, `pet_mr`, `phase_contrast`, `phase_retrieval`, `photoacoustic`, `photometric_stereo`, `polarization`, `polsar`, `portal_imaging`, `proton_radiography`, `proton_therapy_img`, `ptychography`, `pump_probe`, `quantum_illumination`, `radio_astronomy`, `radio_interferometry`, `raman_imaging`, `sar`, `saxs`, `seismic_tomo`, `sem`, `shearography`, `shg`, `sim`, `sims`, `solar_imaging`, `sonar`, `spc`, `spect`, `spect_ct`, `spectral_ct`, `spinning_disk`, `srs`, `sted`, `stem`, `stm`, `streak_camera`, `structured_light`, `swi`, `talbot_lau`, `tem`, `terahertz`, `three_photon`, `tirf`, `tof_camera`, `two_photon`, `ultrasonic_phased_array`, `ultrasound`, `us_mri`, `waxs`, `weather_radar`, `widefield`, `widefield_lowdose`, `xfel_sfx`, `xray_crystallography`, `xray_ndt`, `xray_radiography`, `xrf_imaging`, `xrf_tomo`

## 1. Install PWM

In [ ]:
# Install PWM core package from GitHub
import os
os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"  # Skip large model weights

# Clone the full repo to get algorithm_base
if not os.path.exists("/content/pwm"):
    !git clone --depth 1 https://github.com/integritynoble/Physics_World_Model.git /content/pwm 2>&1 | tail -3

import sys
sys.path.insert(0, "/content/pwm/pwm/public")
sys.path.insert(0, "/content/pwm/pwm/public/packages/pwm_core")

!pip install -q scikit-image scipy h5py 2>&1 | tail -1

print("PWM installed successfully!")

## 2. Configure Modality

In [ ]:
# ============================================================
# CHANGE THIS to any of the 168 modalities listed above
# ============================================================
MODALITY = "ct"  # e.g. "mri", "cassi", "widefield", "photoacoustic"
# ============================================================

# List all available solvers for this modality
from algorithm_base import list_solvers, list_modalities

all_mods = list_modalities()
print(f"Total modalities: {len(all_mods)}")
print(f"\nSolvers for '{MODALITY}':")
for key, info in list_solvers(MODALITY):
    gpu_tag = "[GPU]" if info.get("gpu") else "[CPU]"
    print(f"  {key:30s} {gpu_tag} {info['name']}")
    if info.get('reference'):
        print(f"  {'':30s}       Ref: {info['reference'][:80]}")

## 3. Load Benchmark Data

Load real benchmark data from GCS (`gs://pwm-benchmark-datasets/`) — the same data used at [pwm.platformai.org](https://pwm.platformai.org).

In [ ]:
import numpy as np
import h5py
import os

def load_benchmark_sample(modality, tier="public", sample_idx=0):
    """Load one benchmark sample from GCS or local cache."""
    cache_path = f"/tmp/pwm_benchmark/{modality}/{tier}"
    h5_path = f"{cache_path}/{modality}_challenge_{tier}.h5"
    
    # Download from GCS if not cached
    if not os.path.exists(h5_path):
        os.makedirs(cache_path, exist_ok=True)
        gcs_path = f"gs://pwm-benchmark-datasets/benchmark/{modality}/{tier}/{modality}_challenge_{tier}.h5"
        print(f"Downloading {gcs_path} ...")
        result = os.system(f"gsutil -q cp '{gcs_path}' '{h5_path}' 2>/dev/null")
        if result != 0:
            # Try alternative path
            alt_path = f"gs://pwm-benchmark-datasets/datasets/Benchmark/{modality}/{tier}/{modality}_challenge_{tier}.h5"
            result = os.system(f"gsutil -q cp '{alt_path}' '{h5_path}' 2>/dev/null")
        if result != 0:
            print("GCS download failed — using synthetic data (install gsutil or authenticate for real data)")
            return None
        print(f"Downloaded to {h5_path}")
    
    with h5py.File(h5_path, "r") as f:
        sample_key = f"sample_{sample_idx:02d}"
        s = f[sample_key]
        data = {k: np.array(s[k]) for k in s.keys()}
    return data


def make_synthetic_data(modality):
    """Generate synthetic benchmark-format data for a modality."""
    rng = np.random.RandomState(42)
    
    # Standard 2D case (most modalities)
    x_true = rng.rand(256, 256).astype(np.float32)
    
    # Modality-specific y shapes
    tomo_mods = {"angiography", "brachytherapy_img", "cryo_et", "dexa",
                 "digital_breast_tomo", "flash_lidar", "fluoroscopy", "gpr",
                 "industrial_ct", "muon_tomo", "neutron_tomo", "portal_imaging",
                 "proton_radiography", "proton_therapy_img", "seismic_tomo",
                 "spect", "xray_ndt", "xray_radiography", "xrf_tomo"}
    
    if modality == "ct":
        x_true = rng.rand(362, 362).astype(np.float32)
        y = rng.rand(60, 512).astype(np.float32)
    elif modality == "mri":
        y = (rng.rand(4, 256, 256) + 1j * rng.rand(4, 256, 256)).astype(np.complex64)
    elif modality in tomo_mods:
        y = rng.rand(180, 363).astype(np.float32)
    elif modality == "cacti":
        x_true = rng.rand(256, 256, 8).astype(np.float32)
        y = rng.rand(256, 256).astype(np.float32)
    elif modality == "odt":
        y = rng.rand(60, 256, 256, 2).astype(np.float32)
    elif modality == "phase_retrieval":
        y = rng.rand(8, 256, 256).astype(np.float32)
    elif modality == "raman_imaging":
        y = rng.rand(3, 256, 256).astype(np.float32)
    elif modality == "photoacoustic":
        y = rng.rand(64, 128).astype(np.float32)
    elif modality == "fpm":
        y = rng.rand(39, 64, 64).astype(np.float32)
    elif modality == "ptychography":
        y = rng.rand(100, 64, 64).astype(np.float32)
    elif modality == "cassi":
        x_true = rng.rand(256, 256, 28).astype(np.float32)
        y = rng.rand(256, 256).astype(np.float32)
    else:
        y = rng.rand(256, 256).astype(np.float32)
    
    return {"x_true": x_true, "y": y, "reconstruction_baseline": x_true.mean(axis=-1) if x_true.ndim == 3 else x_true}


# Try to load real data, fall back to synthetic
data = load_benchmark_sample(MODALITY)
if data is None:
    print("Using synthetic data (results won't match leaderboard exactly)")
    data = make_synthetic_data(MODALITY)
else:
    print("Real benchmark data loaded — results match pwm.platformai.org")

print(f"\nData keys: {list(data.keys())}")
for k, v in data.items():
    arr = np.asarray(v)
    print(f"  {k:35s} shape={arr.shape}, dtype={arr.dtype}")

## 4. Extract Measurement Data

In [ ]:
def extract_y(modality, data):
    """Extract measurement data y from benchmark h5 sample."""
    # CT: sinogram
    if "sinogram_ideal" in data:
        return np.asarray(data["sinogram_ideal"], dtype=np.float32)
    # MRI: undersampled k-space
    if "kspace_undersampled" in data:
        return np.asarray(data["kspace_undersampled"])
    # OCT
    if "bscan_ideal" in data:
        return np.asarray(data["bscan_ideal"], dtype=np.float32)
    # Mammography
    if "projection_ideal" in data:
        return np.asarray(data["projection_ideal"], dtype=np.float32)
    # Standard: y
    if "y" in data:
        return np.asarray(data["y"], dtype=np.float32 if np.asarray(data["y"]).dtype != np.complex64 else np.complex64)
    raise ValueError(f"Cannot find measurement data for {modality}. Keys: {list(data.keys())}")

y = extract_y(MODALITY, data)
x_true = np.asarray(data["x_true"], dtype=np.float32)
baseline = np.asarray(data.get("reconstruction_baseline", x_true), dtype=np.float32)

print(f"Measurement y: shape={y.shape}, dtype={y.dtype}")
print(f"Ground truth x_true: shape={x_true.shape}")
print(f"Baseline reconstruction: shape={baseline.shape}")

## 5. Run All Algorithms

In [ ]:
from algorithm_base import run_solver, list_solvers
import time


def psnr(x_hat, x_ref, maxval=None):
    """Compute Peak Signal-to-Noise Ratio."""
    x_hat = np.asarray(x_hat, dtype=np.float64)
    x_ref = np.asarray(x_ref, dtype=np.float64)
    if x_hat.shape != x_ref.shape:
        return float("nan")
    maxval = maxval or float(np.abs(x_ref).max())
    mse = np.mean((x_hat - x_ref) ** 2)
    if mse < 1e-12:
        return 100.0
    return float(20 * np.log10(maxval / np.sqrt(mse)))


def ssim(x_hat, x_ref):
    """Compute Structural Similarity Index."""
    from skimage.metrics import structural_similarity
    x_hat = np.asarray(x_hat, dtype=np.float64)
    x_ref = np.asarray(x_ref, dtype=np.float64)
    if x_hat.shape != x_ref.shape or x_hat.ndim != 2:
        return float("nan")
    data_range = float(x_ref.max() - x_ref.min()) or 1.0
    return float(structural_similarity(x_hat, x_ref, data_range=data_range))


print(f"Running all solvers for '{MODALITY}' ...\n")
results = []

for solver_key, solver_info in list_solvers(MODALITY):
    gpu_only = solver_info.get("gpu", False)
    print(f"  {solver_key:30s} {'[GPU — skipping in CPU-only mode]' if gpu_only else '...'}", end="", flush=True)
    
    # Skip GPU-only solvers if no CUDA
    try:
        import torch
        has_gpu = torch.cuda.is_available()
    except ImportError:
        has_gpu = False
    
    if gpu_only and not has_gpu:
        print()
        results.append({"solver": solver_key, "name": solver_info["name"],
                        "status": "skipped_gpu", "psnr": None, "ssim": None, "time": None})
        continue
    
    t0 = time.time()
    try:
        x_hat = np.asarray(run_solver(MODALITY, solver_key, y), dtype=np.float32)
        elapsed = time.time() - t0
        
        # For 3D outputs, compare against appropriate reference
        ref = x_true
        if x_hat.ndim == 3 and x_true.ndim == 2:
            # Multi-channel output, take mean for comparison
            x_hat_2d = x_hat.mean(axis=-1) if x_hat.shape[-1] <= 32 else x_hat[0]
        elif x_hat.ndim == 2 and x_true.ndim == 3:
            ref = x_true.mean(axis=-1)
            x_hat_2d = x_hat
        else:
            x_hat_2d = x_hat if x_hat.ndim == 2 else x_hat.reshape(x_hat.shape[0], -1).mean(-1) if x_hat.ndim >= 2 else x_hat
            ref = x_true if x_true.ndim == 2 else x_true.mean(axis=-1)
        
        p = psnr(x_hat_2d, ref)
        s = ssim(x_hat_2d, ref)
        
        print(f" PSNR={p:6.1f}dB  SSIM={s:.3f}  ({elapsed:.1f}s)")
        results.append({"solver": solver_key, "name": solver_info["name"],
                        "status": "ok", "psnr": p, "ssim": s, "time": elapsed,
                        "x_hat": x_hat})
    except Exception as e:
        elapsed = time.time() - t0
        print(f" ERROR: {e}")
        results.append({"solver": solver_key, "name": solver_info["name"],
                        "status": "error", "error": str(e), "psnr": None, "ssim": None, "time": elapsed})

print("\nDone!")

## 6. Results Table

In [ ]:
print(f"{'Solver Key':<30} {'Name':<35} {'PSNR':>8} {'SSIM':>7} {'Time':>8} Status")
print("-" * 100)

for r in sorted(results, key=lambda x: (x["psnr"] or -999), reverse=True):
    psnr_str = f"{r['psnr']:8.1f}" if r["psnr"] is not None else "       —"
    ssim_str = f"{r['ssim']:7.3f}" if r["ssim"] is not None else "      —"
    time_str = f"{r['time']:7.1f}s" if r["time"] is not None else "       —"
    status = r["status"]
    print(f"{r['solver']:<30} {r['name']:<35} {psnr_str} {ssim_str} {time_str} {status}")

# Summary
ok_results = [r for r in results if r["status"] == "ok"]
if ok_results:
    best = max(ok_results, key=lambda x: x["psnr"])
    print(f"\nBest CPU result: {best['name']} — PSNR={best['psnr']:.1f}dB, SSIM={best['ssim']:.3f}")
    print(f"Platform URL: https://pwm.platformai.org/benchmark/{MODALITY}")

## 7. Visualize Reconstructions

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def to_display(img):
    """Convert array to displayable 2D image."""
    arr = np.asarray(img, dtype=np.float64)
    if arr.ndim == 3:
        if arr.shape[-1] in (3, 4):  # RGB
            arr = 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]
        elif arr.shape[0] <= 8:  # channel-first spectral
            arr = arr.mean(axis=0)
        else:
            arr = arr.mean(axis=-1)
    # Normalize to [0, 1]
    lo, hi = arr.min(), arr.max()
    if hi - lo > 1e-8:
        arr = (arr - lo) / (hi - lo)
    return arr

ok_results = [r for r in results if r["status"] == "ok" and "x_hat" in r]
n_show = min(len(ok_results), 4)  # Show up to 4 solvers

if n_show == 0:
    print("No successful reconstructions to display.")
else:
    fig, axes = plt.subplots(1, n_show + 2, figsize=(4 * (n_show + 2), 4))
    
    # Ground truth
    axes[0].imshow(to_display(x_true), cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("Ground Truth\n(x_true)")
    axes[0].axis("off")
    
    # Measurement / baseline
    axes[1].imshow(to_display(baseline), cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Baseline Reconstruction")
    axes[1].axis("off")
    
    # Solver results (sorted by PSNR, best first)
    for i, r in enumerate(sorted(ok_results, key=lambda x: x["psnr"], reverse=True)[:n_show]):
        ax = axes[i + 2]
        ax.imshow(to_display(r["x_hat"]), cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{r['name']}\nPSNR={r['psnr']:.1f}dB", fontsize=10)
        ax.axis("off")
    
    plt.suptitle(f"PWM Benchmark: {MODALITY.upper()}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"\nFull leaderboard: https://pwm.platformai.org/benchmark/{MODALITY}")

## 8. (Optional) Verify All 168 Modalities

Run `traditional_cpu` for all 168 modalities and collect PSNR vs baseline.  
This takes ~5-30 minutes depending on the runtime.  Skip this cell if you only need one modality.

In [ ]:
RUN_ALL = False  # Set to True to verify all 168 modalities

if not RUN_ALL:
    print("Skipped. Set RUN_ALL = True to verify all 168 modalities.")
else:
    from algorithm_base import list_modalities, run_solver
    import time
    
    all_results = []
    mods = list_modalities()
    print(f"Verifying {len(mods)} modalities...\n")
    
    for mod_idx, mod in enumerate(mods):
        data = load_benchmark_sample(mod)
        if data is None:
            data = make_synthetic_data(mod)
        
        try:
            y_mod = extract_y(mod, data)
        except Exception as e:
            all_results.append({"modality": mod, "status": "skip", "reason": str(e)})
            continue
        
        x_true_mod = np.asarray(data["x_true"], dtype=np.float32)
        baseline_mod = np.asarray(data.get("reconstruction_baseline", x_true_mod), dtype=np.float32)
        
        t0 = time.time()
        try:
            x_hat = np.asarray(run_solver(mod, "traditional_cpu", y_mod), dtype=np.float32)
            elapsed = time.time() - t0
            
            ref_2d = x_true_mod if x_true_mod.ndim == 2 else x_true_mod.mean(axis=-1)
            hat_2d = x_hat if x_hat.ndim == 2 else x_hat.mean(axis=-1)
            
            shape_ok = (x_hat.shape == x_true_mod.shape)
            p = psnr(hat_2d, ref_2d) if hat_2d.shape == ref_2d.shape else float("nan")
            
            status = "ok" if shape_ok else "shape_mismatch"
            all_results.append({"modality": mod, "status": status, "psnr": p, "time": elapsed})
            print(f"[{mod_idx+1:3d}/{len(mods)}] {mod:35s} {status:15s} PSNR={p:7.1f}dB ({elapsed:.1f}s)")
        except Exception as e:
            all_results.append({"modality": mod, "status": "error", "reason": str(e)[:60]})
            print(f"[{mod_idx+1:3d}/{len(mods)}] {mod:35s} ERROR: {str(e)[:60]}")
    
    # Summary
    ok = [r for r in all_results if r["status"] == "ok"]
    errs = [r for r in all_results if r["status"] == "error"]
    shape_mm = [r for r in all_results if r["status"] == "shape_mismatch"]
    print(f"\n{'='*60}")
    print(f"PASS: {len(ok)}/{len(mods)}, ERRORS: {len(errs)}, SHAPE_MISMATCH: {len(shape_mm)}")
    if ok:
        psnrs = [r["psnr"] for r in ok if r["psnr"] is not None and not np.isnan(r["psnr"])]
        print(f"Median PSNR: {np.median(psnrs):.1f}dB")

## Next Steps

- **Browse the leaderboard**: [pwm.platformai.org/benchmark/{MODALITY}](https://pwm.platformai.org)
- **Submit your algorithm**: See [CONTRIBUTING.md](../CONTRIBUTING.md)
- **Good First Issues**: [GitHub Issues](https://github.com/integritynoble/Physics_World_Model/issues?q=is%3Aissue+is%3Aopen+label%3A%22good+first+issue%22)
- **Discussions**: [GitHub Discussions](https://github.com/integritynoble/Physics_World_Model/discussions)
- **Quickstart**: [PWM_Quickstart.ipynb](PWM_Quickstart.ipynb)